# Chapter 3: Attention Mechanisms — Interactive Demo

This notebook demonstrates the three levels of attention implemented in `attention.py`:
1. **Simple Self-Attention** — manual QKV, no mask
2. **Causal Attention** — with causal mask + dropout
3. **Multi-Head Attention** — parallel heads + output projection

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
import numpy as np

from ch03.attention import SimpleAttention, CausalAttention, MultiHeadAttention

torch.manual_seed(42)

## Setup: Create a simple input

We'll use a small example with 6 tokens and embedding dimension 16.
Think of these as 6 word embeddings in a sentence.

In [ ]:
# Simulated input: 6 tokens, each with 16-dimensional embedding
batch_size = 1
seq_len = 6
d_in = 16
d_out = 16

# Token labels for visualization
tokens = ["The", "cat", "sat", "on", "the", "mat"]

x = torch.randn(batch_size, seq_len, d_in)
print(f"Input shape: {x.shape}  → (batch={batch_size}, seq_len={seq_len}, d_in={d_in})")

## Helper: Visualize attention weights as a heatmap

In [ ]:
def plot_attention(weights, tokens, title="Attention Weights", ax=None):
    """
    Plot attention weights as a heatmap.
    
    Args:
        weights: 2D tensor (seq_len, seq_len)
        tokens: list of token labels
        title: plot title
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    
    w = weights.detach().cpu().float().numpy()
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=w.max())
    
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_yticklabels(tokens)
    ax.set_xlabel('Key (attending to)')
    ax.set_ylabel('Query (from)')
    ax.set_title(title)
    
    # Add text annotations
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            val = w[i, j]
            color = 'white' if val > w.max() * 0.6 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', 
                   color=color, fontsize=8)
    
    return ax

---
## 1. Simple Self-Attention

No causal mask — every token can attend to every other token.
This is **bidirectional** attention (like BERT uses).

In [ ]:
simple_attn = SimpleAttention(d_in, d_out)
ctx_simple, weights_simple = simple_attn(x)

print(f"Context vectors shape: {ctx_simple.shape}")
print(f"Attention weights shape: {weights_simple.shape}")
print(f"\nRow sums (should be 1.0): {weights_simple[0].sum(dim=-1).detach()}")

fig, ax = plt.subplots(figsize=(6, 5))
plot_attention(weights_simple[0], tokens, "Simple Self-Attention (no mask)", ax=ax)
plt.tight_layout()
plt.show()

**Observation:** Every token can attend to every other token — including future tokens.
This is fine for understanding/encoding tasks, but NOT for autoregressive generation.

---
## 2. Causal (Masked) Attention

Adds a causal mask: each token can only attend to itself and previous tokens.
This is what GPT-style models use.

In [ ]:
causal_attn = CausalAttention(d_in, d_out, dropout=0.0)  # dropout=0 for clean visualization
ctx_causal, weights_causal = causal_attn(x)

print(f"Context vectors shape: {ctx_causal.shape}")
print(f"Attention weights shape: {weights_causal.shape}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: the attention weights
plot_attention(weights_causal[0], tokens, "Causal Attention Weights", ax=axes[0])

# Right: show the raw causal mask
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
im = axes[1].imshow(mask, cmap='Reds', vmin=0, vmax=1)
axes[1].set_xticks(range(seq_len))
axes[1].set_yticks(range(seq_len))
axes[1].set_xticklabels(tokens, rotation=45, ha='right')
axes[1].set_yticklabels(tokens)
axes[1].set_title('Causal Mask (red = blocked)')
axes[1].set_xlabel('Key position')
axes[1].set_ylabel('Query position')
for i in range(seq_len):
    for j in range(seq_len):
        val = mask[i, j].item()
        label = '✗' if val > 0 else '✓'
        axes[1].text(j, i, label, ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.show()

**Observation:** The attention weights form a lower-triangular pattern:
- Row 0 ("The"): only attends to itself → weight = 1.0
- Row 1 ("cat"): attends to "The" and "cat" only
- Row 5 ("mat"): attends to all 6 tokens

The red positions in the mask are set to `-inf` before softmax, becoming 0 after.

---
## 3. Multi-Head Attention

Splits the attention into multiple heads, each attending to different patterns.
With `d_out=16` and `num_heads=4`, each head has `head_dim=4`.

In [ ]:
num_heads = 4
mha = MultiHeadAttention(d_in, d_out, num_heads=num_heads, dropout=0.0)
out_mha, weights_mha = mha(x)

print(f"Output shape: {out_mha.shape}")
print(f"Attention weights shape: {weights_mha.shape}  → (batch, num_heads, seq_len, seq_len)")

# Visualize each head's attention pattern
fig, axes = plt.subplots(1, num_heads, figsize=(5 * num_heads, 5))

for h in range(num_heads):
    plot_attention(
        weights_mha[0, h],  # batch 0, head h
        tokens,
        f"Head {h + 1}",
        ax=axes[h]
    )

plt.suptitle("Multi-Head Attention — Each Head Learns Different Patterns", 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Observation:** Each head has a different attention pattern!
- Some heads might focus on adjacent tokens
- Others might focus on specific positions
- The output projection combines all these perspectives

All heads maintain the causal mask (lower-triangular structure).

---
## Comparison: With vs. Without Causal Mask

Let's directly compare how the same input gets different attention patterns.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

plot_attention(weights_simple[0], tokens, "Without Causal Mask\n(bidirectional)", ax=axes[0])
plot_attention(weights_causal[0], tokens, "With Causal Mask\n(autoregressive)", ax=axes[1])

plt.suptitle("Effect of Causal Masking on Attention", fontsize=14)
plt.tight_layout()
plt.show()

---
## Using with Chapter 2 Embeddings

In a real model, attention takes the output of the embedding layer as input.

In [ ]:
import tiktoken
from ch02.embeddings import TokenPositionalEmbedding

# Tokenize a sentence
enc = tiktoken.get_encoding("gpt2")
text = "The cat sat on the mat"
token_ids = enc.encode(text)
real_tokens = [enc.decode([tid]) for tid in token_ids]
print(f"Text: {text}")
print(f"Tokens: {real_tokens}")
print(f"Token IDs: {token_ids}")

# Embed
VOCAB_SIZE = 50257
EMBED_DIM = 64
MAX_LEN = 128

embedding = TokenPositionalEmbedding(VOCAB_SIZE, EMBED_DIM, MAX_LEN)
batch = torch.tensor([token_ids])
embedded = embedding(batch)  # (1, seq_len, 64)
print(f"\nEmbedding output shape: {embedded.shape}")

# Pass through multi-head attention
mha_real = MultiHeadAttention(d_in=EMBED_DIM, d_out=EMBED_DIM, num_heads=4, dropout=0.0)
out, attn_weights = mha_real(embedded)
print(f"Attention output shape: {out.shape}")
print(f"Attention weights shape: {attn_weights.shape}")

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for h in range(4):
    plot_attention(attn_weights[0, h], real_tokens, f"Head {h+1}", ax=axes[h])
plt.suptitle(f'Multi-Head Attention on: "{text}"', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Self-attention** computes a weighted combination of all values, where weights come from query-key similarity
2. **Scaling by √d_k** prevents softmax from saturating with large dimensions
3. **Causal mask** enforces autoregressive (left-to-right) attention for language generation
4. **Multi-head attention** lets the model capture different types of relationships simultaneously
5. All three versions share the same core math: `softmax(QK^T / √d_k) @ V`